# Research06: Filtering 기반 생성 증강 결과 정리

이 노트북은 `Research05` 재실행 결과를 바탕으로, 생성 모델 기반 증강에서 filtering을 적용했을 때의 성능 방향을 다시 정리한다. 특히 최종 결론은 단순 F1 최고값만이 아니라 결함 탐지에서 중요한 `recall` 관점의 trade-off를 함께 반영한다.

사용 기준 파일:

- `data/research05/results/research05_summary.json`
- `data/research05/results/research05_best_by_condition.csv`
- `data/research05/results/research05_best_by_family.csv`
- `data/research05/results/research05_balanced_augmentation_comparison.csv`

주의: 이전 Research02의 element-wise random masking 수치는 사용하지 않고, temporal block 및 feature-group masking으로 재실행된 Research05 결과만 사용한다.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
RESULT_DIR = ROOT / 'data' / 'research05' / 'results'

summary_path = RESULT_DIR / 'research05_summary.json'
best_by_condition_path = RESULT_DIR / 'research05_best_by_condition.csv'
best_by_family_path = RESULT_DIR / 'research05_best_by_family.csv'
comparison_path = RESULT_DIR / 'research05_balanced_augmentation_comparison.csv'

with summary_path.open('r', encoding='utf-8') as f:
    summary = json.load(f)

best_by_condition = pd.read_csv(best_by_condition_path)
best_by_family = pd.read_csv(best_by_family_path)
comparison = pd.read_csv(comparison_path)

metric_cols = ['precision', 'recall', 'f1', 'f2', 'auroc', 'auprc']
display_cols = [
    'condition', 'augmentation_family', 'method',
    'normal_train_used', 'generated_anomaly_used', 'normal_to_anomaly_ratio',
    'threshold', *metric_cols, 'pred_anomaly', 'true_anomaly'
]

## 1. Research05 기준 전체 최고 성능

`best_by_condition` 기준으로 보면 balanced와 imbalanced 조건 모두 전체 최고 F1은 전통 증강 계열에서 나온다. 따라서 전체 성능만 보면 전통 증강이 여전히 강한 baseline이다.

In [2]:
best_by_condition[display_cols].style.format({
    'normal_to_anomaly_ratio': '{:.3f}',
    'threshold': '{:.2f}',
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'f1': '{:.3f}',
    'f2': '{:.3f}',
    'auroc': '{:.3f}',
    'auprc': '{:.3f}',
}).background_gradient(subset=['precision', 'recall', 'f1', 'auprc'], cmap='YlGn')

,condition,augmentation_family,method,normal_train_used,generated_anomaly_used,normal_to_anomaly_ratio,threshold,precision,recall,f1,f2,auroc,auprc,pred_anomaly,true_anomaly
0,balanced,traditional,Frequency domain,1247,1000,1.000,0.90,0.698,0.686,0.692,0.688,0.954,0.729,291,296
1,imbalanced,traditional,Magnitude warping,5000,1000,4.010,0.82,0.964,0.814,0.883,0.840,0.994,0.959,250,296


## 2. Filtering 적용 생성 모델의 위치

생성 모델 계열만 보면 imbalanced 조건에서 `Filtered@750 Masking Diffusion`이 가장 중요한 결과다. 이 방법은 전체 최고 방법인 `Magnitude warping`보다 F1과 AUPRC는 근소하게 낮지만, recall은 더 높다.

즉 결론은 "전통 증강이 완전히 우세"가 아니라, "전통 증강은 F1/AUPRC 우위, filtered 생성 증강은 recall 우위"라는 trade-off로 정리해야 한다.

In [3]:
key_methods = best_by_family[
    (best_by_family['condition'] == 'imbalanced')
    & (
        (best_by_family['augmentation_family'] == 'traditional')
        | (best_by_family['method'] == 'Filtered@750 Masking Diffusion')
        | (best_by_family['method'] == 'Masking Diffusion')
        | (best_by_family['augmentation_family'] == 'none')
    )
].copy()

key_methods = key_methods.sort_values(['f1', 'recall'], ascending=False)
key_methods[display_cols].style.format({
    'normal_to_anomaly_ratio': '{:.3f}',
    'threshold': '{:.2f}',
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'f1': '{:.3f}',
    'f2': '{:.3f}',
    'auroc': '{:.3f}',
    'auprc': '{:.3f}',
}).background_gradient(subset=['precision', 'recall', 'f1', 'auprc'], cmap='Blues')

,condition,augmentation_family,method,normal_train_used,generated_anomaly_used,normal_to_anomaly_ratio,threshold,precision,recall,f1,f2,auroc,auprc,pred_anomaly,true_anomaly
11,imbalanced,traditional,Magnitude warping,5000,1000,4.010,0.82,0.964,0.814,0.883,0.840,0.994,0.959,250,296
7,imbalanced,generative_filtered,Filtered@750 Masking Diffusion,5000,750,5.015,0.85,0.953,0.818,0.880,0.841,0.991,0.944,254,296
6,imbalanced,generative,Masking Diffusion,5000,1000,4.010,0.88,1.000,0.716,0.835,0.759,0.993,0.956,212,296
10,imbalanced,none,Original only,5000,0,20.243,0.91,0.923,0.564,0.700,0.612,0.961,0.810,181,296


In [4]:
traditional = key_methods[key_methods['augmentation_family'] == 'traditional'].iloc[0]
filtered = key_methods[key_methods['method'] == 'Filtered@750 Masking Diffusion'].iloc[0]

delta = pd.DataFrame([
    {
        'comparison': 'Filtered@750 Masking Diffusion - Magnitude warping',
        'delta_precision': filtered['precision'] - traditional['precision'],
        'delta_recall': filtered['recall'] - traditional['recall'],
        'delta_f1': filtered['f1'] - traditional['f1'],
        'delta_f2': filtered['f2'] - traditional['f2'],
        'delta_auprc': filtered['auprc'] - traditional['auprc'],
        'delta_pred_anomaly': filtered['pred_anomaly'] - traditional['pred_anomaly'],
    }
])

delta.style.format({
    'delta_precision': '{:+.4f}',
    'delta_recall': '{:+.4f}',
    'delta_f1': '{:+.4f}',
    'delta_f2': '{:+.4f}',
    'delta_auprc': '{:+.4f}',
    'delta_pred_anomaly': '{:+.0f}',
})

,comparison,delta_precision,delta_recall,delta_f1,delta_f2,delta_auprc,delta_pred_anomaly
0,Filtered@750 Masking Diffusion - Magnitude warping,-0.0112,+0.0034,-0.0028,+0.0011,-0.0151,+4


## 3. Filtering 계열 전체 비교

Filtering 계열은 생성 샘플을 그대로 모두 쓰는 방식보다, 생성 샘플의 품질이나 anomaly 확률을 기준으로 선별한 뒤 classifier training에 사용하는 방식이다. Research05 summary 기준 filtering은 다음 두 방향으로 정의된다.

- `Filtered`: real anomaly centroid와의 근접성, normal centroid와의 분리도, real-anomaly feature range 일관성을 기준으로 생성 window를 선별한다.
- `ScoreFiltered`: real-data-only RandomForest가 부여한 anomaly probability가 높은 생성 window를 선별한다.

결과적으로 imbalanced 조건에서 filtering은 생성 모델 성능을 전통 증강과 거의 같은 수준까지 끌어올렸고, 특히 `Filtered@750 Masking Diffusion`은 recall 기준으로 가장 좋은 비교 포인트가 된다.

In [5]:
filtering_families = ['generative_filtered', 'generative_score_filtered', 'generative_hybrid']

filtering_rank = comparison[
    (comparison['condition'] == 'imbalanced')
    & (comparison['augmentation_family'].isin(filtering_families))
].copy()

filtering_rank = filtering_rank.sort_values(['recall', 'f1', 'auprc'], ascending=False)
filtering_rank[display_cols].head(12).style.format({
    'normal_to_anomaly_ratio': '{:.3f}',
    'threshold': '{:.2f}',
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'f1': '{:.3f}',
    'f2': '{:.3f}',
    'auroc': '{:.3f}',
    'auprc': '{:.3f}',
}).background_gradient(subset=['recall', 'f1', 'auprc'], cmap='YlOrRd')

,condition,augmentation_family,method,normal_train_used,generated_anomaly_used,normal_to_anomaly_ratio,threshold,precision,recall,f1,f2,auroc,auprc,pred_anomaly,true_anomaly
44,imbalanced,generative_filtered,Filtered@750 Masking Diffusion,5000,750,5.015,0.85,0.953,0.818,0.880,0.841,0.991,0.944,254,296
45,imbalanced,generative_filtered,Filtered@950 Masking Diffusion,5000,950,4.177,0.86,0.930,0.807,0.864,0.829,0.991,0.941,257,296
46,imbalanced,generative_filtered,Filtered@900 Masking Diffusion,5000,900,4.359,0.87,0.951,0.784,0.859,0.812,0.989,0.938,244,296
62,imbalanced,generative_filtered,Filtered@750 Diffusion,5000,750,5.015,0.84,0.680,0.784,0.728,0.761,0.975,0.844,341,296
59,imbalanced,generative_score_filtered,ScoreFiltered@750 Diffusion,5000,750,5.015,0.86,0.716,0.767,0.741,0.756,0.978,0.845,317,296
74,imbalanced,generative_filtered,Filtered@900 Diffusion,5000,900,4.359,0.86,0.653,0.764,0.704,0.739,0.973,0.821,346,296
53,imbalanced,generative_filtered,Filtered@750 Masking GT-GAN,5000,750,5.015,0.86,0.820,0.753,0.785,0.766,0.984,0.898,272,296
54,imbalanced,generative_score_filtered,ScoreFiltered@950 Masking Diffusion,5000,950,4.177,0.89,0.820,0.753,0.785,0.766,0.983,0.895,272,296
63,imbalanced,generative_filtered,Filtered@950 Diffusion,5000,950,4.177,0.85,0.705,0.750,0.727,0.740,0.974,0.812,315,296
57,imbalanced,generative_filtered,Filtered@900 GT-GAN,5000,900,4.359,0.84,0.784,0.736,0.760,0.746,0.974,0.830,278,296


## 4. Balanced 조건 해석

Balanced 조건에서는 생성 모델 filtering의 이점이 imbalanced 조건만큼 강하지 않다. 일부 방법은 recall을 높이지만 precision이 크게 낮아져 F1이 떨어진다. 이는 단순히 class ratio를 1:1로 맞추는 것이 항상 최종 성능을 높이지는 않으며, 충분한 normal sample을 유지한 imbalanced 학습 조건이 실제 test 분포에 더 잘 맞았을 가능성을 보여준다.

In [6]:
balanced_focus = best_by_family[
    (best_by_family['condition'] == 'balanced')
    & best_by_family['augmentation_family'].isin(['traditional', 'generative', 'generative_filtered', 'generative_score_filtered', 'generative_hybrid', 'none'])
].copy()

balanced_focus = balanced_focus.sort_values(['f1', 'recall'], ascending=False)
balanced_focus[display_cols].style.format({
    'normal_to_anomaly_ratio': '{:.3f}',
    'threshold': '{:.2f}',
    'precision': '{:.3f}',
    'recall': '{:.3f}',
    'f1': '{:.3f}',
    'f2': '{:.3f}',
    'auroc': '{:.3f}',
    'auprc': '{:.3f}',
}).background_gradient(subset=['precision', 'recall', 'f1', 'auprc'], cmap='Purples')

,condition,augmentation_family,method,normal_train_used,generated_anomaly_used,normal_to_anomaly_ratio,threshold,precision,recall,f1,f2,auroc,auprc,pred_anomaly,true_anomaly
5,balanced,traditional,Frequency domain,1247,1000,1.000,0.90,0.698,0.686,0.692,0.688,0.954,0.729,291,296
1,balanced,generative_filtered,Filtered@900 GT-GAN,1147,900,1.000,0.92,0.689,0.568,0.622,0.588,0.931,0.671,244,296
3,balanced,generative_score_filtered,ScoreFiltered@750 Diffusion,997,750,1.000,0.91,0.550,0.686,0.611,0.654,0.933,0.680,369,296
0,balanced,generative,Masking Diffusion,1247,1000,1.000,0.94,0.574,0.645,0.607,0.630,0.942,0.730,333,296
2,balanced,generative_hybrid,Hybrid ScoreFiltered@750 Masking Diffusion + Weak Magnitude,997,750,1.000,0.87,0.379,0.713,0.495,0.606,0.927,0.616,556,296
4,balanced,none,Original only,247,0,1.000,0.95,0.239,0.794,0.368,0.542,0.838,0.351,982,296


## 5. 최종 결론 작성 방향

최종 결론은 다음 방향으로 정리한다.

1. 전체 최고 성능은 imbalanced 조건의 전통 증강 `Magnitude warping`이다. 이 방법은 F1 `0.8828`, AUPRC `0.9590`으로 가장 높은 종합 성능을 보였다.
2. 그러나 filtered 생성 모델인 `Filtered@750 Masking Diffusion`은 recall `0.8176`으로 `Magnitude warping`의 recall `0.8142`보다 높다.
3. 따라서 생성 모델 기반 증강은 F1/AUPRC 기준으로 전통 증강을 명확히 넘지는 못했지만, 결함을 놓치지 않는 recall 관점에서는 filtering 적용 후 전통 증강보다 유리한 지점이 확인되었다.
4. 이 결과는 생성 샘플 수를 단순히 늘리는 것보다, 생성 샘플의 품질을 선별하고 실제 학습 분포와 class ratio를 함께 관리하는 것이 중요하다는 방향을 지지한다.
5. Balanced 조건에서는 일부 생성/filtered 방법의 recall이 높아질 수 있지만 precision 저하가 커져 F1이 낮아지는 경향이 있다. 최종 보고서에서는 balanced 조건보다 imbalanced 조건에서의 filtered 생성 모델 효과를 핵심 근거로 제시하는 것이 적절하다.

보고서 문장 예시:

> In the rerun Research05 results, traditional augmentation achieved the best overall F1 and AUPRC under the imbalanced setting, with Magnitude warping reaching F1 = 0.8828 and AUPRC = 0.9590. However, the quality-filtered generative method, Filtered@750 Masking Diffusion, achieved a slightly higher recall of 0.8176 compared with 0.8142 for Magnitude warping. This indicates that filtered generative augmentation does not clearly dominate traditional augmentation in overall score, but it provides a meaningful advantage for recall-oriented defect detection, where missing anomalies is more costly than increasing false positives.